In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np
import itertools
from sklearn.linear_model import PassiveAggressiveClassifier

In [5]:
df = pd.read_csv('fake_or_real_news.csv') # Load data into DataFrame

df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [8]:
y = df.label
y = df.label
X_train, X_test, y_train, y_test = train_test_split(df['text'], y, test_size=0.33,random_state=53)

## Understanding CountVectorizer

The CountVectorizer provides a simple way to both tokenize a collection of text documents and build a vocabulary of known words, but also to encode new documents using that vocabulary.

You can use it as follows:

- Create an instance of the CountVectorizer class.
- Call the fit() function in order to learn a vocabulary from one or more documents.
- Call the transform() function on one or more documents as needed to encode each as a vector.

An encoded vector is returned with a length of the entire vocabulary and an integer count for the number of times each word appeared in the document.

Because these vectors will contain a lot of zeros, we call them sparse. Python provides an efficient way of handling sparse vectors in the scipy.sparse package.

In [9]:
# Simple example

from sklearn.feature_extraction.text import CountVectorizer

# list of text documents
text = ["The quick brown fox jumped over the lazy dog."]
# create the transform
vectorizer = CountVectorizer()
# tokenize and build vocab
vectorizer.fit(text)
# summarize
print(vectorizer.vocabulary_)
# encode document
vector = vectorizer.transform(text)
# summarize encoded vector
print(vector.shape)
print(type(vector))
print(vector.toarray())


{'the': 7, 'quick': 6, 'brown': 0, 'fox': 2, 'jumped': 3, 'over': 5, 'lazy': 4, 'dog': 1}
(1, 8)
<class 'scipy.sparse._csr.csr_matrix'>
[[1 1 1 1 1 1 1 2]]


In [13]:
corpus = [
'This is the first document.',
'This document is the second document.',
'And this is the third one.',
'Is this the first document?',
]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
print(vectorizer.get_feature_names_out())
print(X.toarray()) 

['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']
[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]


## use count_vectorizer on fake news dataset

In [11]:
count_vectorizer = CountVectorizer(stop_words='english')
count_train = count_vectorizer.fit_transform(X_train.values)
count_test = count_vectorizer.transform(X_test.values)

## Understanding TfidfVectorizer Using a Simple Example

The TfidfVectorizer will tokenize documents, learn the vocabulary and inverse document frequency weightings, and allow you to encode new documents. Alternately, if you already have a learned CountVectorizer, you can use it with a TfidfTransformer to just calculate the inverse document frequencies and start encoding documents.

The same create, fit, and transform process is used as with the CountVectorizer.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# list of text documents
text = ["The quick brown fox jumped over the lazy dog.","The dog.", "The fox"]

# create the transform
vectorizer = TfidfVectorizer()

# tokenize and build vocab
vectorizer.fit(text)

# summarize
print("Vocabulary :- ",vectorizer.vocabulary_)
print("IDF :- ",vectorizer.idf_)

# encode document
vector = vectorizer.transform([text[0]])

# summarize encoded vector
print("Text : ", text[0])
print("Shape : ",vector.shape)

print("Representation : ", vector.toarray())

Vocabulary :-  {'the': 7, 'quick': 6, 'brown': 0, 'fox': 2, 'jumped': 3, 'over': 5, 'lazy': 4, 'dog': 1}
IDF :-  [1.69314718 1.28768207 1.28768207 1.69314718 1.69314718 1.69314718
 1.69314718 1.        ]
Text :  The quick brown fox jumped over the lazy dog.
Shape :  (1, 8)
Representation :  [[0.36388646 0.27674503 0.27674503 0.36388646 0.36388646 0.36388646
  0.36388646 0.42983441]]


## Using Tf-IDF Vectorizer on Fake News Dataset

In [16]:
# Initialize the `tfidf_vectorizer` 
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7) 

# Fit and transform the training data 
tfidf_train = tfidf_vectorizer.fit_transform(X_train) 

# Transform the test set 
tfidf_test = tfidf_vectorizer.transform(X_test)

print(tfidf_test)

  (0, 56196)	0.043317993781946
  (0, 56091)	0.032700334892684514
  (0, 55858)	0.05420932672571138
  (0, 55358)	0.059873902121258926
  (0, 55027)	0.048465128413932454
  (0, 54772)	0.06742854646276102
  (0, 54647)	0.03727533728426692
  (0, 54484)	0.1271579409566499
  (0, 54400)	0.05200707680397371
  (0, 54182)	0.17039563890103226
  (0, 52193)	0.050614987002584974
  (0, 52166)	0.09946716362549408
  (0, 52164)	0.07141989873688631
  (0, 51896)	0.09946458588236584
  (0, 51083)	0.06988430196901438
  (0, 50973)	0.11421142130023298
  (0, 50920)	0.08108261495679815
  (0, 50712)	0.07153047197062776
  (0, 50690)	0.05024667107829908
  (0, 50627)	0.04286648720912277
  (0, 48965)	0.06202757667895662
  (0, 48929)	0.17556869825083593
  (0, 46631)	0.14220349264725846
  (0, 46621)	0.10706347107534141
  (0, 44522)	0.040932482324428275
  :	:
  (2090, 5969)	0.03303772830203347
  (2090, 5576)	0.04943418930560652
  (2090, 5530)	0.05752451982231977
  (2090, 4919)	0.026792001261175008
  (2090, 4321)	0.038896002

In [17]:
# Get the feature names of `tfidf_vectorizer` 
print(tfidf_vectorizer.get_feature_names_out()[-10:])
# Get the feature names of `count_vectorizer` 
print(count_vectorizer.get_feature_names_out()[0:10])

['حلب' 'عربي' 'عن' 'لم' 'ما' 'محاولات' 'من' 'هذا' 'والمرضى' 'ยงade']
['00' '000' '0000' '00000031' '000035' '00006' '0001' '0001pt' '000ft'
 '000km']


In [24]:
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

# Initialize a Naive Bayes classifier
nb_classifier = MultinomialNB()

# Train the classifier
nb_classifier.fit(tfidf_train, y_train)

# Predict on the test data
tfidf_pred = nb_classifier.predict(tfidf_test)

# Calculate accuracy, precision, recall, and F1-score
accuracy = metrics.accuracy_score(y_test, tfidf_pred)
precision = metrics.precision_score(y_test, tfidf_pred, pos_label='FAKE')
recall = metrics.recall_score(y_test, tfidf_pred, pos_label='FAKE')
f1 = metrics.f1_score(y_test, tfidf_pred, pos_label='FAKE')


print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")


Accuracy: 0.8565279770444764
Precision: 0.9597402597402598
Recall: 0.7331349206349206
F1 Score: 0.8312710911136107


In [27]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

# Create a pipeline with CountVectorizer and RandomForestClassifier
pipeline = Pipeline([
    ('vect', CountVectorizer(stop_words='english')),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Predict on the test data
y_pred = pipeline.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='FAKE')
recall = recall_score(y_test, y_pred, pos_label='FAKE')
f1 = f1_score(y_test, y_pred, pos_label='FAKE')

# Print metrics
print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')


Accuracy: 0.9052880820836622
Precision: 0.9191419141914191
Recall: 0.8869426751592356
F1 Score: 0.9027552674230146


In [30]:
# Get the feature names from the CountVectorizer
feature_names = pipeline.named_steps['vect'].get_feature_names_out()

# Get the feature importances from the RandomForestClassifier
importances = pipeline.named_steps['clf'].feature_importances_

# Combine feature names and their importances
features_importance = zip(feature_names, importances)

# Convert to a list and sort them by importance
sorted_features = sorted(features_importance, key=lambda x: x[1], reverse=True)

# Display the most important features
print("Most important features:")
for feature, importance in sorted_features[:20]:  # top 20 features
    print(f"{feature}: {importance}")

Most important features:
said: 0.012029805059367378
2016: 0.009464778473789952
october: 0.007299163844958564
sen: 0.007069820535119567
obama: 0.006829905640177653
ted: 0.006481914179415151
republican: 0.006035712604413448
cruz: 0.005285186456862267
com: 0.004982563346806398
gop: 0.004965363678883874
senate: 0.0046754795304959
president: 0.0037880227363558936
republicans: 0.0035300635260791247
candidates: 0.0034358297529496784
cnn: 0.0032978459251329414
house: 0.0032336857947761377
campaign: 0.0030685523513053663
fox: 0.003034860817825821
article: 0.002974304894960841
hillary: 0.0029683243775723922
